# rv32i-core: A Verified Single-Cycle RV32I RISC-V Core

Run the whole project here -- nothing is installed on your own machine.
This clones the repo, installs Icarus Verilog / Yosys / Verilator, runs the
full test suite against the RTL, synthesizes the control/datapath logic to
generic gates, and regenerates every figure in `results/`.

Press **Runtime -> Run all**, top to bottom.

Repository: https://github.com/fatinnihal532-hub/rv32i-core

## Setup

In [ ]:
!git clone -q https://github.com/fatinnihal532-hub/rv32i-core.git
%cd rv32i-core
!apt-get -qq update && apt-get -qq install -y iverilog yosys verilator > /dev/null
!pip install -q -r requirements.txt

## Run the RTL test suite (Icarus Verilog)
30 instruction-level tests plus two full test programs (a Fibonacci-style recurrence and a bubble sort), each checked against an independent Python model.

In [ ]:
!python3 -m unittest discover -s tests -v

## Lint the RTL with Verilator

In [ ]:
!verilator --lint-only -Wall src/*.v

## Synthesize the control/datapath logic to generic gates (Yosys)
Excludes instruction/data memory -- see `docs/methodology.md` for why.

In [ ]:
!mkdir -p results && rm -f results/synth_stat.txt
!yosys -q -s scripts/synth.tcl
!python3 scripts/report_synth.py

## Check the core against hand-computed and modelled results

In [ ]:
!python3 verify.py

## Rebuild every figure in `results/`

In [ ]:
!python3 scripts/make_figures.py

In [ ]:
from IPython.display import SVG, display
for name in ['datapath', 'gate_counts']:
    display(SVG(f'results/{name}.svg'))

## Assemble and run your own program
Anything built from the instructions `assembler/tiny_asm.py` supports (see its docstring).

In [ ]:
import sys
sys.path.insert(0, 'tests')
from harness import run_program

program = '''
li t0, 7
li t1, 6
add t2, t0, t1
sw t2, 0(zero)
ecall
'''
st = run_program(program)
print('t2 =', st['regs'][7], ' mem[0] =', st['mem'][0])